In [1]:
import pyspark.sql.functions as f
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer, Bucketizer
from pyspark.ml import Pipeline
from pyspark.ml.regression import GeneralizedLinearRegression, RandomForestRegressor, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
from pathlib import Path
path = Path.cwd()
if not Path(path, "data").exists(): path = path.parent

In [3]:
import sys
from pathlib import Path
root = Path.cwd().parent 
sys.path.append(str(root))

from minio_utils import MinioSparkClient
import os
spark = MinioSparkClient(endpoint=os.getenv("MINIO_ENDPOINT", "").replace("http://", "").replace("https://", ""),
                        access_key=os.getenv("MINIO_ACCESS_KEY"),
                        secret_key=os.getenv("MINIO_SECRET_KEY"),
                        memory = 16,
                        heapsize = 8,
                        num_part = 2000,
                        bucket_name="pd2",
                        base_dir="cityenjoyer")
                        
spark.connect()

## Carga y división en train, test

In [4]:
df_tip = spark.read_parquet("tip_model_data/*")
df_tip.show(5)

+--------+------------------+------------+------------+------------+----------------+----------------+------------+------------+---+----+-----+----------+
|VendorID|trip_distance_cbrt|payment_type|PUAskingRent|DOAskingRent|PUlandmark_count|DOlandmark_count|PUis_airport|DOis_airport|dow|hour|month|tip_amount|
+--------+------------------+------------+------------+------------+----------------+----------------+------------+------------+---+----+-----+----------+
|       0|14.665174320150205|           1|      6300.0|      3995.0|              24|              24|           0|           0|  4|  16|   10|       407|
|       0|15.710955610759562|           1|      3995.0|      5088.5|              24|              18|           0|           0|  4|  16|   10|       100|
|       0|11.313305651645821|           1|      3902.5|      3902.5|               8|               8|           0|           0|  4|  16|   10|       100|
|       0|14.094597464129784|           1|      3902.5|      4595.0|  

In [6]:
df_tip.groupBy("VendorID").count().collect()

[Row(VendorID=3, count=64285883),
 Row(VendorID=1, count=534380),
 Row(VendorID=2, count=166473591),
 Row(VendorID=0, count=43995121)]

In [6]:
df_tip.select(f.min("tip_amount"), f.max("tip_amount")).collect()

[Row(min(tip_amount)=0, max(tip_amount)=10000)]

In [ ]:
df_tip = df_tip.filter("VendorID == 0").sample(.1, 42)

In [ ]:
train_df, test_df = df_tip.randomSplit([.7,.3], seed=42)

train_df.cache()
test_df.cache()

print(f"Datos de entrenamiento: {train_df.count()} registros")
print(f"Datos de prueba: {test_df.count()} registros")

In [ ]:
train_df.select(['tip_amount']).describe().show()

+-------+------------------+
|summary|        tip_amount|
+-------+------------------+
|  count|         192697621|
|   mean|132.42502610865134|
| stddev| 316.3332411644987|
|    min|                 0|
|    max|             10000|
+-------+------------------+



In [ ]:
test_df.select(['tip_amount']).describe().show()

+-------+------------------+
|summary|        tip_amount|
+-------+------------------+
|  count|          82591354|
|   mean|132.42229351270836|
| stddev| 316.3248080998809|
|    min|                 0|
|    max|             10000|
+-------+------------------+



## Crear flujo de procesamiento

In [ ]:
categorical_cols = ["payment_type", "PUis_airport", "DOis_airport", "dow", "hour", "month"] # "VendorID", 
numeric_cols = ["trip_distance_cbrt", "PUAskingRent", "DOAskingRent", "PUlandmark_count", "DOlandmark_count"]
objective = "tip_amount"

columnas_indexadas = [f"{c}_idx" for c in categorical_cols]
indexer = StringIndexer(
    inputCols=categorical_cols,
    outputCols=columnas_indexadas,
    handleInvalid="keep"
)

columnas_one_hot = [f"{c}_ohe" for c in categorical_cols]
encoder = OneHotEncoder(
    inputCols=columnas_indexadas,
    outputCols=columnas_one_hot,
    dropLast=True
)

VectorAssembler_cdfce4e51a5a

## Modelo GLM
Probamos con un GLM con el parámetro gamma ya que la variable objetivo veíamos que era bastante asimétrica hacia la derecha.

In [ ]:
feature_cols = columnas_one_hot + numeric_cols
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
assembler

In [ ]:
glm = GeneralizedLinearRegression(
    featuresCol="features", 
    labelCol=objective, 
    family="tweedie",
    variancePower=1.5, # 1.5 es el estándar para datos con ceros y asimetría
    linkPower=1
)

glm_pipeline = Pipeline(stages=[
    indexer, encoder, bucketizer, assembler, glm
])
print("Pipeline GLM creado con", len(glm_pipeline.getStages()), "etapas")

Pipeline GLM creado con 5 etapas


In [9]:
glm_model = glm_pipeline.fit(train_df)
glm_predictions = glm_model.transform(test_df)

# El fallo es posible que esté relacionado con linkPower o que haya que escalar la variable objetivo
print("Primeras predicciones:")
glm_predictions.select(objective, "features", "prediction").show(10)

Primeras predicciones:
+----------+--------------------+----------+
|tip_amount|            features|prediction|
+----------+--------------------+----------+
|         0|(56,[1,5,7,11,18,...|   1.0E-16|
|         0|(56,[1,5,7,12,16,...|   1.0E-16|
|         0|(56,[1,5,7,13,30,...|   1.0E-16|
|         0|(56,[1,5,7,10,19,...|   1.0E-16|
|         0|(56,[1,5,7,10,16,...|   1.0E-16|
|         0|(56,[1,5,7,13,16,...|   1.0E-16|
|         0|(56,[1,5,7,15,17,...|   1.0E-16|
|         0|(56,[1,5,7,11,17,...|   1.0E-16|
|         0|(56,[1,5,7,11,19,...|   1.0E-16|
|         0|(56,[1,5,7,12,18,...|   1.0E-16|
+----------+--------------------+----------+
only showing top 10 rows



In [10]:
lr_model = glm_model.stages[-1]

# Obtener coeficientes (es un DenseVector)
coefficients = lr_model.coefficients
intercept = lr_model.intercept

# Convertir a array de NumPy para trabajar más fácil
import numpy as np
coef_array = coefficients.toArray()

print(f"Intercepto: {intercept}")
print(f"Coeficientes: {coef_array}")

Intercepto: -70.97696653738254
Coeficientes: [ 7.62375925e+02  1.63224602e+02 -6.75958433e+01 -4.38631571e+01
 -1.81595587e+01 -1.35967591e+02  1.35967591e+02 -1.45665832e+02
  1.45665832e+02 -3.29871806e+01  2.63394614e+01 -1.13435195e+01
  3.13142096e+01  1.05999717e+01 -2.17838483e+01  1.34164650e+01
  1.16057839e+01  4.31494192e+01 -2.64719126e+01  7.01982226e+01
 -1.18136111e+01  3.15099268e+01 -1.03523268e+01 -2.54685139e+01
  2.55911260e+01  2.85833321e+01  2.92810163e+01  2.95475460e+01
 -3.75422088e+01  2.87600844e+01  2.84247479e+01  2.81467640e+01
 -6.19528195e+01  3.28132643e+00 -1.04331326e+02 -5.32908707e+01
 -1.45711568e+02 -6.45110193e+01 -4.37173981e+01 -7.61959650e+01
  1.49685304e+01 -4.09622320e+00  1.45811760e+00 -1.54677450e+01
 -7.30538338e+00  3.38218424e+01  2.59748565e+01 -2.20996980e+01
  4.67663096e+00 -3.08260149e+01  1.47060349e+01  1.18955249e+01
  2.80820647e-02  1.84418692e-02  1.95337950e-01 -2.65032864e+00]


In [11]:
rmse_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="r2"
)

glm_rmse = rmse_evaluator.evaluate(glm_predictions)
glm_mae = mae_evaluator.evaluate(glm_predictions)
glm_r2 = r2_evaluator.evaluate(glm_predictions)

print('Métricas test')
print(f"RMSE (Root Mean Squared Error): {glm_rmse/100:.2f} $")
print(f"MAE (Mean Absolute Error):      {glm_mae/100:.2f} $")
print(f"R2 (R-squared):                 {glm_r2:.4f}")

Métricas test
RMSE (Root Mean Squared Error): 4.16 $
MAE (Mean Absolute Error):      3.36 $
R2 (R-squared):                 -0.2514


In [ ]:
glm_model.write().overwrite().save(str(path / "data" / "glm_model_v1"))

## Decision Tree

In [ ]:
feature_cols_tree = columnas_indexadas + columnas_bucket + numeric_cols
assembler_tree = VectorAssembler(
    inputCols=feature_cols_tree,
    outputCol="features"
)
assembler_tree

VectorAssembler_4b9f06eabeeb

In [ ]:
dt = DecisionTreeRegressor(
    featuresCol="features", 
    labelCol=objective, 
    maxDepth=10
)

dt_pipeline = Pipeline(stages=[
    indexer, bucketizer, assembler_tree, dt
])

print("Entrenando un único Árbol de Decisión...")
dt_model = dt_pipeline.fit(train_df)
dt_predictions = dt_model.transform(test_df)

dt_predictions.select(objective, "features", "prediction").show(10)

Entrenando un único Árbol de Decisión...
+----------+--------------------+-----------------+
|tip_amount|            features|       prediction|
+----------+--------------------+-----------------+
|         0|(13,[0,3,4,5,9,10...|11.34951225315251|
|         0|(13,[0,3,5,7,9,10...|2.961854738020407|
|         0|[1.0,0.0,0.0,4.0,...|11.34951225315251|
|         0|(13,[0,3,4,5,9,10...|11.34951225315251|
|       100|[1.0,0.0,0.0,3.0,...|11.34951225315251|
|         0|(13,[0,3,5,6,9,10...|2.961854738020407|
|         0|[1.0,0.0,0.0,2.0,...|2.961854738020407|
|         0|(13,[0,3,5,6,9,10...|2.961854738020407|
|         0|[1.0,0.0,0.0,2.0,...|2.961854738020407|
|         0|[1.0,0.0,0.0,1.0,...|2.961854738020407|
+----------+--------------------+-----------------+
only showing top 10 rows



In [ ]:
rmse_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="r2"
)

dt_rmse = rmse_evaluator.evaluate(dt_predictions)
dt_mae = mae_evaluator.evaluate(dt_predictions)
dt_r2 = r2_evaluator.evaluate(dt_predictions)

print('Métricas test')
print(f"RMSE (Root Mean Squared Error): {dt_rmse / 100:.2f} $")
print(f"MAE (Mean Absolute Error):      {dt_mae / 100:.2f} $")
print(f"R2 (R-squared):                 {dt_r2:.4f}")

Métricas test
RMSE (Root Mean Squared Error): 2.08 $
MAE (Mean Absolute Error):      0.22 $
R2 (R-squared):                 0.0159


## Random Forest

In [8]:
feature_cols_tree = columnas_indexadas + numeric_cols # + columnas_bucket
assembler_tree = VectorAssembler(
    inputCols=feature_cols_tree,
    outputCol="features"
)
assembler_tree

VectorAssembler_320bcbb5b665

In [9]:
rf = RandomForestRegressor(
    featuresCol="features", 
    labelCol=objective, 
    numTrees=50,
    maxDepth=6,
    seed=42
)

rf_pipeline = Pipeline(stages=[
    indexer, bucketizer, assembler_tree, rf
])

print("Pipeline de Random Forest creado con", len(rf_pipeline.getStages()), "etapas")

# 3. Entrenamos y predecimos
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

print("Primeras predicciones con Random Forest:")
rf_predictions.select(objective, "features", "prediction").show(10)

Pipeline de Random Forest creado con 4 etapas
Primeras predicciones con Random Forest:
+----------+--------------------+------------------+
|tip_amount|            features|        prediction|
+----------+--------------------+------------------+
|         0|(11,[0,3,4,5,7,8]...| 21.35340541841351|
|         0|(11,[0,3,5,7,8,10...|16.289978595413928|
|         0|[1.0,0.0,0.0,4.0,...|23.726798542563166|
|         0|(11,[0,3,4,5,7,8]...|21.719672280816145|
|       100|[1.0,0.0,0.0,3.0,...| 25.29002273859874|
|         0|(11,[0,3,5,7,8,9]...|21.971262094517307|
|         0|[1.0,0.0,0.0,2.0,...|23.251684387771256|
|         0|(11,[0,3,5,7,8,9]...| 21.60325840037864|
|         0|[1.0,0.0,0.0,2.0,...|  15.0054834494676|
|         0|[1.0,0.0,0.0,1.0,...| 16.65780702541919|
+----------+--------------------+------------------+
only showing top 10 rows



In [10]:
rmse_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=objective,
    predictionCol="prediction",
    metricName="r2"
)

rf_rmse = rmse_evaluator.evaluate(rf_predictions)
rf_mae = mae_evaluator.evaluate(rf_predictions)
rf_r2 = r2_evaluator.evaluate(rf_predictions)

print('Métricas test')
print(f"RMSE (Root Mean Squared Error): {rf_rmse / 100:.2f} $")
print(f"MAE (Mean Absolute Error):      {rf_mae / 100:.2f} $")
print(f"R2 (R-squared):                 {rf_r2:.4f}")

Métricas test
RMSE (Root Mean Squared Error): 2.30 $
MAE (Mean Absolute Error):      1.21 $
R2 (R-squared):                 0.6172


Este es el mejor resultado hasta ahora (15/03/2026 17:27)

In [ ]:
rf_model.write().overwrite().save(str(path / "data" / "rf_model_v1"))